In [ ]:
storage_account = "stretailbidevgk01"
container = "retail-data"

storage_key = dbutils.secrets.get(
    scope="kv-retail-bi-dev",
    key="adls-storage-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("ADLS authentication configured successfully")

In [ ]:
# ============================================================
# 06_Auto_Loader
# Retail Business Intelligence Data Platform
# ============================================================

from pyspark.sql.functions import (
    current_timestamp,
    input_file_name
)

# ADLS Gen2 storage
storage_account = "stretailbidevgk01"
container = "retail-data"

base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net"

landing_path = f"{base_path}/landing"
bronze_path = f"{base_path}/bronze"
checkpoint_path = f"{base_path}/audit/checkpoints"

print("Landing Path:", landing_path)
print("Bronze Path:", bronze_path)
print("Checkpoint Path:", checkpoint_path)

In [ ]:
# ============================================================
# 1. Auto Loader - Orders
# ============================================================

orders_landing_path = f"{landing_path}/wwi/orders"
orders_bronze_path = f"{bronze_path}/wwi/orders"
orders_checkpoint_path = f"{checkpoint_path}/orders"

df_orders_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", orders_checkpoint_path)
        .option("header", "true")
        .load(orders_landing_path)
)

df_orders_bronze.printSchema()

In [ ]:
# Remove the schema/checkpoint created from the incorrect inference

dbutils.fs.rm(orders_checkpoint_path, True)

print("Removed:", orders_checkpoint_path)

In [ ]:
display(
    spark.read
        .format("csv")
        .option("header", "false")
        .option("inferSchema", "false")
        .load(orders_landing_path)
        .limit(5)
)

In [ ]:
"""Orders schema"""

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    TimestampType,
    BooleanType
)

orders_schema = StructType([
    StructField("OrderID", IntegerType(), True),
    StructField("CustomerID", IntegerType(), True),
    StructField("SalespersonPersonID", IntegerType(), True),
    StructField("PickedByPersonID", IntegerType(), True),
    StructField("ContactPersonID", IntegerType(), True),
    StructField("OrderDate", TimestampType(), True),
    StructField("ExpectedDeliveryDate", TimestampType(), True),
    StructField("CustomerPurchaseOrderNumber", IntegerType(), True),
    StructField("IsUndersupplyBackordered", BooleanType(), True),
    StructField("PickingCompletedWhen", TimestampType(), True),
    StructField("Comments", StringType(), True),
    StructField("DeliveryInstructions", StringType(), True),
    StructField("InternalComments", StringType(), True),
    StructField("TotalDryItems", IntegerType(), True),
    StructField("TotalChillerItems", IntegerType(), True),
    StructField("LastEditedWhen", TimestampType(), True)
])

print("Orders schema defined.")

In [ ]:
# ============================================================
# 2. Auto Loader - Orders with Explicit Schema
# ============================================================

df_orders_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", orders_checkpoint_path)
        .option("header", "false")
        .option("cloudFiles.inferColumnTypes", "false")
        .schema(orders_schema)
        .load(orders_landing_path)
)

df_orders_bronze.printSchema()

In [ ]:
# ============================================================
# 3. Add Auto Loader ingestion metadata
# Unity Catalog compatible
# ============================================================

from pyspark.sql.functions import col, current_timestamp

df_orders_bronze_enriched = (
    df_orders_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

display(
    df_orders_bronze_enriched,
    checkpointLocation=orders_checkpoint_path
)

In [ ]:
# ============================================================
# 4. Production checkpoint locations
# ============================================================

orders_schema_path = f"{checkpoint_path}/orders/schema"
orders_stream_checkpoint = f"{checkpoint_path}/orders/stream"

print("Schema Location:", orders_schema_path)
print("Stream Checkpoint:", orders_stream_checkpoint)

In [ ]:
# ============================================================
# 5. Recreate Orders Auto Loader stream
#    with production checkpoint structure
# ============================================================

df_orders_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", orders_schema_path)
        .option("header", "false")
        .option("cloudFiles.inferColumnTypes", "false")
        .schema(orders_schema)
        .load(orders_landing_path)
)

df_orders_bronze_enriched = (
    df_orders_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

df_orders_bronze_enriched.printSchema()

In [ ]:
# ============================================================
# 6. Auto Loader -> Bronze Delta
# ============================================================

orders_bronze_query = (
    df_orders_bronze_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", orders_stream_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(orders_bronze_path)
)

orders_bronze_query.awaitTermination()

print("Orders Bronze ingestion completed.")

In [ ]:
# ============================================================
# 7. Validate Orders Bronze
# ============================================================

df_orders_bronze_check = (
    spark.read
        .format("delta")
        .load(orders_bronze_path)
)

print("Bronze row count:", df_orders_bronze_check.count())
print("Bronze column count:", len(df_orders_bronze_check.columns))

display(
    df_orders_bronze_check
        .orderBy("OrderID")
        .limit(10)
)

In [ ]:
# ============================================================
# 8. Bronze Data Quality Checks
# ============================================================

print("Null OrderID:",
      df_orders_bronze_check.filter("OrderID IS NULL").count())

print("Duplicate OrderID:",
      df_orders_bronze_check.groupBy("OrderID")
          .count()
          .filter("count > 1")
          .count())

print("Null _ingested_at:",
      df_orders_bronze_check.filter("_ingested_at IS NULL").count())

print("Null _source_file:",
      df_orders_bronze_check.filter("_source_file IS NULL").count())

In [ ]:
# ============================================================
# 9. OrderLines Schema
# ============================================================
from pyspark.sql.types import DecimalType
orderlines_schema = StructType([
    StructField("OrderLineID", IntegerType(), True),
    StructField("OrderID", IntegerType(), True),
    StructField("StockItemID", IntegerType(), True),
    StructField("Description", StringType(), True),
    StructField("PackageTypeID", IntegerType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("UnitPrice", DecimalType(18, 2), True),
    StructField("TaxRate", DecimalType(18, 3), True),
    StructField("PickedQuantity", IntegerType(), True),
    StructField("PickingCompletedWhen", TimestampType(), True),
    StructField("LastEditedBy", IntegerType(), True),
    StructField("LastEditedWhen", TimestampType(), True)
])

print("OrderLines schema defined.")

In [ ]:
# ============================================================
# 10. OrderLines paths
# ============================================================

orderlines_landing_path = f"{landing_path}/wwi/orderlines"
orderlines_bronze_path = f"{bronze_path}/wwi/orderlines"

orderlines_schema_path = f"{checkpoint_path}/orderlines/schema"
orderlines_stream_checkpoint = f"{checkpoint_path}/orderlines/stream"

print("Landing:", orderlines_landing_path)
print("Bronze:", orderlines_bronze_path)
print("Schema:", orderlines_schema_path)
print("Stream:", orderlines_stream_checkpoint)

In [ ]:
# ============================================================
# 11. Auto Loader - OrderLines
# ============================================================

df_orderlines_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", orderlines_schema_path)
        .option("header", "false")
        .option("cloudFiles.inferColumnTypes", "false")
        .schema(orderlines_schema)
        .load(orderlines_landing_path)
)

df_orderlines_bronze_enriched = (
    df_orderlines_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

df_orderlines_bronze_enriched.printSchema()

In [ ]:
# ============================================================
# 12. Auto Loader -> Bronze Delta - OrderLines
# ============================================================

orderlines_bronze_query = (
    df_orderlines_bronze_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", orderlines_stream_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(orderlines_bronze_path)
)

orderlines_bronze_query.awaitTermination()

print("OrderLines Bronze ingestion completed.")

In [ ]:
# ============================================================
# 13. Validate OrderLines Bronze
# ============================================================

df_orderlines_bronze_check = (
    spark.read
        .format("delta")
        .load(orderlines_bronze_path)
)

print("Bronze row count:", df_orderlines_bronze_check.count())
print("Bronze column count:", len(df_orderlines_bronze_check.columns))

print(
    "Null OrderLineID:",
    df_orderlines_bronze_check.filter("OrderLineID IS NULL").count()
)

print(
    "Duplicate OrderLineID:",
    df_orderlines_bronze_check
        .groupBy("OrderLineID")
        .count()
        .filter("count > 1")
        .count()
)

print(
    "Null _ingested_at:",
    df_orderlines_bronze_check.filter("_ingested_at IS NULL").count()
)

print(
    "Null _source_file:",
    df_orderlines_bronze_check.filter("_source_file IS NULL").count()
)

In [ ]:
# ============================================================
# 14. Customers Schema
# ============================================================

customers_schema = StructType([
    StructField("CustomerID", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("BillToCustomerID", IntegerType(), True),
    StructField("CustomerCategoryID", IntegerType(), True),
    StructField("BuyingGroupID", IntegerType(), True),
    StructField("PrimaryContactPersonID", IntegerType(), True),
    StructField("AlternateContactPersonID", IntegerType(), True),
    StructField("DeliveryMethodID", IntegerType(), True),
    StructField("DeliveryCityID", IntegerType(), True),
    StructField("PostalCityID", IntegerType(), True),
    StructField("CreditLimit", DecimalType(18, 2), True),
    StructField("AccountOpenedDate", TimestampType(), True),
    StructField("StandardDiscountPercentage", DecimalType(18, 3), True),
    StructField("IsStatementSent", BooleanType(), True),
    StructField("IsOnCreditHold", BooleanType(), True),
    StructField("PaymentDays", IntegerType(), True),
    StructField("PhoneNumber", StringType(), True),
    StructField("FaxNumber", StringType(), True),
    StructField("DeliveryRun", StringType(), True),
    StructField("RunPosition", IntegerType(), True),
    StructField("WebsiteURL", StringType(), True),
    StructField("DeliveryAddressLine1", StringType(), True),
    StructField("DeliveryAddressLine2", StringType(), True),
    StructField("DeliveryPostalCode", StringType(), True),
    StructField("DeliveryLocation", StringType(), True),
    StructField("PostalAddressLine1", StringType(), True),
    StructField("PostalAddressLine2", StringType(), True),
    StructField("PostalPostalCode", StringType(), True),
    StructField("LastEditedBy", IntegerType(), True),
    StructField("ValidFrom", TimestampType(), True),
    StructField("ValidTo", TimestampType(), True)
])

print("Customers schema defined.")

In [ ]:
# ============================================================
# 15. Customers paths
# ============================================================

customers_landing_path = f"{landing_path}/wwi/customers"
customers_bronze_path = f"{bronze_path}/wwi/customers"

customers_schema_path = f"{checkpoint_path}/customers/schema"
customers_stream_checkpoint = f"{checkpoint_path}/customers/stream"

print("Landing:", customers_landing_path)
print("Bronze:", customers_bronze_path)
print("Schema:", customers_schema_path)
print("Stream:", customers_stream_checkpoint)

In [ ]:
# ============================================================
# 16. Auto Loader - Customers
# ============================================================

df_customers_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", customers_schema_path)
        .option("header", "false")
        .option("cloudFiles.inferColumnTypes", "false")
        .schema(customers_schema)
        .load(customers_landing_path)
)

df_customers_bronze_enriched = (
    df_customers_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

df_customers_bronze_enriched.printSchema()

In [ ]:
# ============================================================
# 17. Auto Loader -> Bronze Delta - Customers
# ============================================================

customers_bronze_query = (
    df_customers_bronze_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", customers_stream_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(customers_bronze_path)
)

customers_bronze_query.awaitTermination()

print("Customers Bronze ingestion completed.")

In [ ]:
# ============================================================
# 18. Validate Customers Bronze
# ============================================================

df_customers_bronze_check = (
    spark.read
        .format("delta")
        .load(customers_bronze_path)
)

print("Bronze row count:", df_customers_bronze_check.count())
print("Bronze column count:", len(df_customers_bronze_check.columns))

print(
    "Null CustomerID:",
    df_customers_bronze_check.filter("CustomerID IS NULL").count()
)

print(
    "Duplicate CustomerID:",
    df_customers_bronze_check
        .groupBy("CustomerID")
        .count()
        .filter("count > 1")
        .count()
)

print(
    "Null _ingested_at:",
    df_customers_bronze_check.filter("_ingested_at IS NULL").count()
)

print(
    "Null _source_file:",
    df_customers_bronze_check.filter("_source_file IS NULL").count()
)

In [ ]:
# ============================================================
# 19. Weather paths
# ============================================================

weather_landing_path = f"{landing_path}/weather"
weather_bronze_path = f"{bronze_path}/weather"

weather_schema_path = f"{checkpoint_path}/weather/schema"
weather_stream_checkpoint = f"{checkpoint_path}/weather/stream"

print("Landing:", weather_landing_path)
print("Bronze:", weather_bronze_path)
print("Schema:", weather_schema_path)
print("Stream:", weather_stream_checkpoint)

In [ ]:
# ============================================================
# 20. Inspect Weather Landing JSON
# ============================================================

df_weather_raw = (
    spark.read
        .format("json")
        .load(weather_landing_path)
)

df_weather_raw.printSchema()

display(df_weather_raw.limit(5))

In [ ]:
# ============================================================
# 21. Weather Bronze Schema
# ============================================================
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    DoubleType,
    LongType,
    StringType
)
weather_schema = StructType([
    StructField(
        "daily",
        StructType([
            StructField("precipitation_sum", ArrayType(DoubleType()), True),
            StructField("rain_sum", ArrayType(DoubleType()), True),
            StructField("temperature_2m_max", ArrayType(DoubleType()), True),
            StructField("temperature_2m_min", ArrayType(DoubleType()), True),
            StructField("time", ArrayType(StringType()), True),
            StructField("wind_speed_10m_max", ArrayType(DoubleType()), True)
        ]),
        True
    ),

    StructField(
        "daily_units",
        StructType([
            StructField("precipitation_sum", StringType(), True),
            StructField("rain_sum", StringType(), True),
            StructField("temperature_2m_max", StringType(), True),
            StructField("temperature_2m_min", StringType(), True),
            StructField("time", StringType(), True),
            StructField("wind_speed_10m_max", StringType(), True)
        ]),
        True
    ),

    StructField("elevation", DoubleType(), True),
    StructField("generationtime_ms", DoubleType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("timezone", StringType(), True),
    StructField("timezone_abbreviation", StringType(), True),
    StructField("utc_offset_seconds", LongType(), True)
])

print("Weather Bronze schema defined.")

In [ ]:
# ============================================================
# 22. Auto Loader - Weather
# ============================================================

df_weather_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", weather_schema_path)
        .schema(weather_schema)
        .load(weather_landing_path)
)

df_weather_bronze_enriched = (
    df_weather_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

df_weather_bronze_enriched.printSchema()

In [ ]:
# ============================================================
# 23. Auto Loader -> Bronze Delta - Weather
# ============================================================

weather_bronze_query = (
    df_weather_bronze_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", weather_stream_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(weather_bronze_path)
)

weather_bronze_query.awaitTermination()

print("Weather Bronze ingestion completed.")

In [ ]:
# ============================================================
# 24. Validate Weather Bronze
# ============================================================

df_weather_bronze_check = (
    spark.read
        .format("delta")
        .load(weather_bronze_path)
)

print("Bronze row count:", df_weather_bronze_check.count())
print("Bronze column count:", len(df_weather_bronze_check.columns))

print(
    "Null latitude:",
    df_weather_bronze_check.filter("latitude IS NULL").count()
)

print(
    "Null longitude:",
    df_weather_bronze_check.filter("longitude IS NULL").count()
)

print(
    "Null _ingested_at:",
    df_weather_bronze_check.filter("_ingested_at IS NULL").count()
)

print(
    "Null _source_file:",
    df_weather_bronze_check.filter("_source_file IS NULL").count()
)

In [ ]:
# ============================================================
# 25. Inspect Weather Bronze metadata
# ============================================================

display(
    df_weather_bronze_check
    .select(
        "latitude",
        "longitude",
        "timezone",
        "_ingested_at",
        "_source_file"
    )
    .orderBy("_ingested_at")
)

In [ ]:
# Count records by metadata status

df_weather_bronze_check.select(
    "_ingested_at",
    "_source_file"
).groupBy(
    "_ingested_at",
    "_source_file"
).count().orderBy("_ingested_at").show(truncate=False)

In [ ]:
display(
    df_weather_bronze_check
    .select(
        "daily.time",
        "latitude",
        "longitude",
        "timezone",
        "_ingested_at",
        "_source_file"
    )
    .orderBy("_ingested_at")
)

In [ ]:
dbutils.fs.rm(weather_bronze_path, True)
dbutils.fs.rm(weather_stream_checkpoint, True)

print("Weather Bronze and stream checkpoint cleared.")

In [ ]:
df_weather_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", weather_schema_path)
        .schema(weather_schema)
        .load(weather_landing_path)
)

df_weather_bronze_enriched = (
    df_weather_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

weather_bronze_query = (
    df_weather_bronze_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", weather_stream_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(weather_bronze_path)
)

weather_bronze_query.awaitTermination()

In [ ]:
df_weather_bronze_check = spark.read.format("delta").load(weather_bronze_path)

print("Rows:", df_weather_bronze_check.count())

display(
    df_weather_bronze_check.select(
        "daily.time",
        "latitude",
        "longitude",
        "timezone",
        "_ingested_at",
        "_source_file"
    )
)

Inspect the actual Holiday JSON

In [ ]:
holiday_landing_path = f"{landing_path}/holiday"

df_holiday_raw = (
    spark.read
        .format("json")
        .load(holiday_landing_path)
)

df_holiday_raw.printSchema()
display(df_holiday_raw.limit(10))

In [ ]:
from pyspark.sql.types import (
    StructType,
    StructField,
    ArrayType,
    StringType,
    BooleanType
)

holiday_schema = StructType([
    StructField("counties", ArrayType(StringType()), True),
    StructField("countryCode", StringType(), True),
    StructField("date", StringType(), True),
    StructField("fixed", BooleanType(), True),
    StructField("global", BooleanType(), True),
    StructField("launchYear", StringType(), True),
    StructField("localName", StringType(), True),
    StructField("name", StringType(), True),
    StructField("types", ArrayType(StringType()), True)
])

In [ ]:
holiday_landing_path = f"{landing_path}/holiday"
holiday_bronze_path = f"{bronze_path}/holiday"

holiday_schema_path = f"{checkpoint_path}/holiday/schema"
holiday_stream_checkpoint = f"{checkpoint_path}/holiday/stream"

In [ ]:
from pyspark.sql.functions import col
df_holiday_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", holiday_schema_path)
        .schema(holiday_schema)
        .load(holiday_landing_path)
)

df_holiday_bronze_enriched = (
    df_holiday_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

In [ ]:
holiday_bronze_query = (
    df_holiday_bronze_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", holiday_stream_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(holiday_bronze_path)
)

holiday_bronze_query.awaitTermination()

In [ ]:
df_holiday_bronze_check = (
    spark.read
        .format("delta")
        .load(holiday_bronze_path)
)

print("Holiday Bronze rows:", df_holiday_bronze_check.count())
print("Holiday Bronze columns:", len(df_holiday_bronze_check.columns))

display(
    df_holiday_bronze_check.select(
        "date",
        "countryCode",
        "localName",
        "name",
        "types",
        "_ingested_at",
        "_source_file"
    )
)

Clean Holiday Bronze

In [ ]:
dbutils.fs.rm(holiday_bronze_path, True)
dbutils.fs.rm(holiday_stream_checkpoint, True)

print("Holiday Bronze and stream checkpoint cleared.")

Re-run the Holiday Auto Loader write

In [ ]:
df_holiday_bronze = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", holiday_schema_path)
        .schema(holiday_schema)
        .load(holiday_landing_path)
)

df_holiday_bronze_enriched = (
    df_holiday_bronze
    .select(
        "*",
        current_timestamp().alias("_ingested_at"),
        col("_metadata.file_path").alias("_source_file")
    )
)

holiday_bronze_query = (
    df_holiday_bronze_enriched.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", holiday_stream_checkpoint)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .start(holiday_bronze_path)
)

holiday_bronze_query.awaitTermination()

Validate

In [ ]:
df_holiday_bronze_check = (
    spark.read
        .format("delta")
        .load(holiday_bronze_path)
)

print("Rows:", df_holiday_bronze_check.count())
print("Columns:", len(df_holiday_bronze_check.columns))

display(
    df_holiday_bronze_check.select(
        "date",
        "countryCode",
        "localName",
        "name",
        "types",
        "_ingested_at",
        "_source_file"
    )
)